# Notebook 4: rioxarray — Geospatial xarray

Open a COG with rioxarray, inspect CRS/bounds, and optionally clip or reproject.

**Dependencies:** `rioxarray`, `rasterio`, `pystac-client`, `planetary-computer`

In [ ]:
import rioxarray
import pystac_client
import planetary_computer

## Get a signed COG URL (Sentinel-2 B04)

In [ ]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)
search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=[-122.4, 37.6, -122.2, 37.8],
    datetime="2023-06-15",
    max_items=1,
)
item = planetary_computer.sign(list(search.items())[0])
url = item.assets["B04"].href

## Open as xarray DataArray

`open_rasterio` returns a DataArray with dimensions (band, y, x) and spatial metadata on `.rio`.

In [ ]:
data = rioxarray.open_rasterio(url)
print("Dimensions:", data.dims)
print("CRS:", data.rio.crs)
print("Bounds:", data.rio.bounds)
print("Shape:", data.shape)

## Drop band dimension (single band)

If only one band, squeeze and drop the band dim for simpler indexing.

In [ ]:
data = data.squeeze("band", drop=True)
data

## Clip to a bounding box

Read only the window that intersects the box (streaming).

In [ ]:
clipped = data.rio.clip_box(
    minx=-122.35, miny=37.65, maxx=-122.28, maxy=37.72
)
print("Clipped shape:", clipped.shape)
print("Clipped bounds:", clipped.rio.bounds)

## Open with chunking (Dask-backed)

Use `chunks=` so the array is lazy; useful for large rasters and Dask workflows.

In [ ]:
data_chunked = rioxarray.open_rasterio(url, chunks={"x": 1024, "y": 1024})
print("Chunks:", data_chunked.chunks)
print("Dask?", hasattr(data_chunked.data, "compute"))

## Optional: quick plot

Plot the clipped region (triggers read for that window).

In [ ]:
import matplotlib.pyplot as plt

clipped.plot(cmap="gray", figsize=(6, 5))
plt.title("rioxarray — COG clip (B04)")
plt.tight_layout()
plt.show()